# Social Impact Analysis: Impacto de Redes Sociales en Estudiantes

## Introducción del Proyecto

Este notebook implementa un sistema analítico orquestado por skills para analizar el impacto de las redes sociales en estudiantes. El sistema sigue una arquitectura modular basada en el patrón **Agente Orquestador con Skills**, donde cada skill tiene una responsabilidad específica:

- **EDA Skill**: Análisis exploratorio de datos
- **Correlation Skill**: Análisis de correlaciones y relaciones
- **Insight Skill**: Generación de insights y recomendaciones

### Objetivo General

Analizar cómo el uso de redes sociales afecta la salud mental, el sueño y el rendimiento académico de los estudiantes, identificando patrones, correlaciones significativas y generando recomendaciones basadas en evidencia cuantitativa.

### Variables Principales del Dataset

- `Age`: Edad del estudiante
- `Gender`: Género
- `Academic_Level`: Nivel académico
- `Country`: País
- `Avg_Daily_Usage_Hours`: Horas promedio de uso diario de redes sociales
- `Most_Used_Platform`: Plataforma más utilizada
- `Affects_Academic_Performance`: Si afecta el rendimiento académico (Yes/No)
- `Sleep_Hours_Per_Night`: Horas de sueño por noche
- `Mental_Health_Score`: Puntuación de salud mental (0-10)
- `Overall_Impact`: Impacto general (Positive/Neutral/Negative)

## 1. Configuración e Importación de Librerías

El agente orquestador inicializa el entorno de análisis importando las librerías necesarias para ejecutar los skills analíticos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from scipy import stats

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Configuración para mostrar gráficos inline
%matplotlib inline

# Configuración de tamaño de figuras
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✓ Librerías importadas correctamente")
print("✓ Configuración de visualización completada")

## 2. Carga y Validación del Dataset

El agente carga el dataset CSV y realiza una validación inicial para verificar la calidad de los datos antes de ejecutar los skills analíticos.

In [ ]:
# Definir ruta del dataset
data_path = Path('Data set.csv')

# Cargar dataset
df = pd.read_csv(data_path, sep=';')

print(f"✓ Dataset cargado exitosamente")
print(f"✓ Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"✓ Tamaño en memoria: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

### 2.1 Vista General del Dataset

El agente realiza una inspección inicial para comprender la estructura del dataset y verificar la integridad de los datos.

In [ ]:
# Mostrar primeras filas
print("=== Primeras 5 filas del dataset ===")
display(df.head())

# Información del dataset
print("\n=== Información del dataset ===")
df.info()

In [ ]:
# Estadísticas descriptivas
print("=== Estadísticas Descriptivas ===")
display(df.describe())

# Verificar valores nulos
print("\n=== Valores Nulos por Columna ===")
null_counts = df.isnull().sum()
null_percentages = (df.isnull().sum() / len(df) * 100).round(2)
null_info = pd.DataFrame({
    'Nulos': null_counts,
    'Porcentaje': null_percentages
})
display(null_info)

In [ ]:
# Verificar duplicados
duplicates = df.duplicated().sum()
print(f"=== Registros Duplicados ===")
print(f"Total de duplicados: {duplicates}")

# Tipos de datos
print("\n=== Tipos de Datos ===")
display(df.dtypes)

## 3. Contexto del Agente Orquestador

El sistema sigue la arquitectura definida en `agent.md`, que establece un pipeline secuencial de ejecución de skills:

```
Dataset CSV ➔ EDA Skill ➔ Correlation Skill ➔ Insight Skill ➔ Conclusiones
```

### Skills del Sistema:

1. **EDA Skill**: Análisis exploratorio con distribuciones, correlaciones y comparaciones por categoría
2. **Correlation Skill**: Identificación de relaciones significativas entre variables
3. **Insight Skill**: Síntesis de resultados y generación de conclusiones accionables

### Umbrales de Alerta Definidos:

- Uso > 6h: Alto consumo
- Salud mental < 5: Deterioro
- Sueño < 6h: Privación
- Correlación uso-salud < -0.5: Relación preocupante

## 4. Skill EDA: Análisis Exploratorio de Datos

### Objetivo

Analizar distribuciones, patrones y relaciones iniciales en el dataset de impacto de redes sociales. El agente identifica:

- distribuciones relevantes de variables numéricas
- posibles valores atípicos
- patrones iniciales de comportamiento
- diferencias entre grupos por género y plataforma

### 4.1 Distribuciones de Variables Numéricas

El agente analiza las principales variables numéricas para identificar distribuciones, asimetrías y posibles valores atípicos.

In [ ]:
# Seleccionar variables numéricas
numeric_cols = ['Age', 'Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night', 'Mental_Health_Score']

# Crear grid de histogramas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribuciones de Variables Numéricas', fontsize=16, fontweight='bold')

axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    
    # Histograma con KDE
    sns.histplot(data=df, x=col, kde=True, ax=ax, color='steelblue', alpha=0.7)
    
    # Añadir líneas de media y mediana
    mean_val = df[col].mean()
    median_val = df[col].median()
    
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Media: {mean_val:.2f}')
    ax.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Mediana: {median_val:.2f}')
    
    ax.set_xlabel(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('Frecuencia', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Interpretación
print("\n=== Interpretación de Distribuciones ===")
for col in numeric_cols:
    mean_val = df[col].mean()
    median_val = df[col].median()
    std_val = df[col].std()
    skewness = df[col].skew()
    
    print(f"\n{col}:")
    print(f"  - Media: {mean_val:.2f}")
    print(f"  - Mediana: {median_val:.2f}")
    print(f"  - Desviación Estándar: {std_val:.2f}")
    print(f"  - Asimetría: {skewness:.2f}")
    
    if skewness > 0.5:
        print(f"  - Distribución sesgada a la derecha")
    elif skewness < -0.5:
        print(f"  - Distribución sesgada a la izquierda")
    else:
        print(f"  - Distribución aproximadamente simétrica")

### 4.2 Boxplots para Detección de Outliers

El agente utiliza el método IQR (Interquartile Range) para detectar posibles valores atípicos en las variables numéricas.

In [ ]:
# Crear boxplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Boxplots para Detección de Outliers', fontsize=16, fontweight='bold')

axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    sns.boxplot(data=df, y=col, ax=ax, color='lightcoral')
    ax.set_ylabel(col, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Detección de outliers usando IQR
print("\n=== Detección de Outliers (Método IQR) ===")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    print(f"\n{col}:")
    print(f"  - Límite inferior: {lower_bound:.2f}")
    print(f"  - Límite superior: {upper_bound:.2f}")
    print(f"  - Outliers detectados: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

### 4.3 Comparaciones por Categoría

El agente compara la salud mental entre diferentes grupos para identificar posibles segmentos vulnerables.

In [ ]:
# Boxplot de Salud Mental por Plataforma
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Salud Mental por Categoría', fontsize=16, fontweight='bold')

# Por plataforma
sns.boxplot(data=df, x='Most_Used_Platform', y='Mental_Health_Score', ax=axes[0])
axes[0].set_xlabel('Plataforma', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Puntuación de Salud Mental', fontsize=11, fontweight='bold')
axes[0].set_title('Salud Mental por Plataforma', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# Por género
sns.boxplot(data=df, x='Gender', y='Mental_Health_Score', ax=axes[1])
axes[1].set_xlabel('Género', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Puntuación de Salud Mental', fontsize=11, fontweight='bold')
axes[1].set_title('Salud Mental por Género', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Estadísticas por plataforma
print("\n=== Estadísticas de Salud Mental por Plataforma ===")
platform_stats = df.groupby('Most_Used_Platform')['Mental_Health_Score'].agg(['mean', 'median', 'std', 'count']).round(2)
platform_stats = platform_stats.sort_values('mean', ascending=False)
display(platform_stats)

### 4.4 Relaciones Bivariadas

El agente explora relaciones entre variables clave mediante scatter plots con líneas de tendencia.

In [ ]:
# Crear scatter plots bivariados
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Relaciones Bivariadas', fontsize=16, fontweight='bold')

# Uso vs Salud Mental
sns.scatterplot(data=df, x='Avg_Daily_Usage_Hours', y='Mental_Health_Score', 
                alpha=0.6, ax=axes[0,0], color='steelblue')
axes[0,0].set_xlabel('Uso Diario (horas)', fontsize=11, fontweight='bold')
axes[0,0].set_ylabel('Salud Mental', fontsize=11, fontweight='bold')
axes[0,0].set_title('Uso vs Salud Mental', fontsize=12, fontweight='bold')
axes[0,0].grid(True, alpha=0.3)

# Añadir línea de tendencia
z = np.polyfit(df['Avg_Daily_Usage_Hours'], df['Mental_Health_Score'], 1)
p = np.poly1d(z)
axes[0,0].plot(df['Avg_Daily_Usage_Hours'], p(df['Avg_Daily_Usage_Hours']), 
              "r--", alpha=0.8, linewidth=2, label=f'Tendencia: y={z[0]:.2f}x+{z[1]:.2f}')
axes[0,0].legend()

# Uso vs Sueño
sns.scatterplot(data=df, x='Avg_Daily_Usage_Hours', y='Sleep_Hours_Per_Night', 
                alpha=0.6, ax=axes[0,1], color='darkgreen')
axes[0,1].set_xlabel('Uso Diario (horas)', fontsize=11, fontweight='bold')
axes[0,1].set_ylabel('Horas de Sueño', fontsize=11, fontweight='bold')
axes[0,1].set_title('Uso vs Sueño', fontsize=12, fontweight='bold')
axes[0,1].grid(True, alpha=0.3)

# Línea de tendencia
z2 = np.polyfit(df['Avg_Daily_Usage_Hours'], df['Sleep_Hours_Per_Night'], 1)
p2 = np.poly1d(z2)
axes[0,1].plot(df['Avg_Daily_Usage_Hours'], p2(df['Avg_Daily_Usage_Hours']), 
               "r--", alpha=0.8, linewidth=2, label=f'Tendencia: y={z2[0]:.2f}x+{z2[1]:.2f}')
axes[0,1].legend()

# Sueño vs Salud Mental
sns.scatterplot(data=df, x='Sleep_Hours_Per_Night', y='Mental_Health_Score', 
                alpha=0.6, ax=axes[1,0], color='purple')
axes[1,0].set_xlabel('Horas de Sueño', fontsize=11, fontweight='bold')
axes[1,0].set_ylabel('Salud Mental', fontsize=11, fontweight='bold')
axes[1,0].set_title('Sueño vs Salud Mental', fontsize=12, fontweight='bold')
axes[1,0].grid(True, alpha=0.3)

# Línea de tendencia
z3 = np.polyfit(df['Sleep_Hours_Per_Night'], df['Mental_Health_Score'], 1)
p3 = np.poly1d(z3)
axes[1,0].plot(df['Sleep_Hours_Per_Night'], p3(df['Sleep_Hours_Per_Night']), 
               "r--", alpha=0.8, linewidth=2, label=f'Tendencia: y={z3[0]:.2f}x+{z3[1]:.2f}')
axes[1,0].legend()

# Edad vs Uso
sns.scatterplot(data=df, x='Age', y='Avg_Daily_Usage_Hours', 
                alpha=0.6, ax=axes[1,1], color='orange')
axes[1,1].set_xlabel('Edad', fontsize=11, fontweight='bold')
axes[1,1].set_ylabel('Uso Diario (horas)', fontsize=11, fontweight='bold')
axes[1,1].set_title('Edad vs Uso', fontsize=12, fontweight='bold')
axes[1,1].grid(True, alpha=0.3)

# Línea de tendencia
z4 = np.polyfit(df['Age'], df['Avg_Daily_Usage_Hours'], 1)
p4 = np.poly1d(z4)
axes[1,1].plot(df['Age'], p4(df['Age']), 
               "r--", alpha=0.8, linewidth=2, label=f'Tendencia: y={z4[0]:.2f}x+{z4[1]:.2f}')
axes[1,1].legend()

plt.tight_layout()
plt.show()

# Cálculo de correlaciones para scatter plots
print("\n=== Correlaciones para Relaciones Bivariadas ===")
corr_usage_mental = df['Avg_Daily_Usage_Hours'].corr(df['Mental_Health_Score'])
corr_usage_sleep = df['Avg_Daily_Usage_Hours'].corr(df['Sleep_Hours_Per_Night'])
corr_sleep_mental = df['Sleep_Hours_Per_Night'].corr(df['Mental_Health_Score'])
corr_age_usage = df['Age'].corr(df['Avg_Daily_Usage_Hours'])

print(f"Uso vs Salud Mental: {corr_usage_mental:.3f}")
print(f"Uso vs Sueño: {corr_usage_sleep:.3f}")
print(f"Sueño vs Salud Mental: {corr_sleep_mental:.3f}")
print(f"Edad vs Uso: {corr_age_usage:.3f}")

### 4.5 Interpretación de Alertas

El agente genera alertas automáticas si se detectan indicadores de riesgo asociados al bienestar estudiantil.

In [ ]:
# Análisis de umbrales de alerta
print("=== Análisis de Umbrales de Alerta ===")

# Umbral: Uso > 6h (Alto consumo)
high_usage = df[df['Avg_Daily_Usage_Hours'] > 6]
print(f"\n🔴 Alto consumo (>6h): {len(high_usage)} estudiantes ({len(high_usage)/len(df)*100:.1f}%)")

# Umbral: Salud mental < 5 (Deterioro)
poor_mental = df[df['Mental_Health_Score'] < 5]
print(f"🔴 Deterioro salud mental (<5): {len(poor_mental)} estudiantes ({len(poor_mental)/len(df)*100:.1f}%)")

# Umbral: Sueño < 6h (Privación)
sleep_deprivation = df[df['Sleep_Hours_Per_Night'] < 6]
print(f"🔴 Privación de sueño (<6h): {len(sleep_deprivation)} estudiantes ({len(sleep_deprivation)/len(df)*100:.1f}%)")

# Intersección de riesgos
high_risk = df[(df['Avg_Daily_Usage_Hours'] > 6) & (df['Mental_Health_Score'] < 5)]
print(f"\n⚠️ Alto riesgo (Uso>6h AND Salud<5): {len(high_risk)} estudiantes ({len(high_risk)/len(df)*100:.1f}%)")

# Afectación académica
academic_impact = df[df['Affects_Academic_Performance'] == 'Yes']
print(f"🔴 Afectación académica: {len(academic_impact)} estudiantes ({len(academic_impact)/len(df)*100:.1f}%)")

## 5. Skill Correlation Analysis: Análisis de Correlaciones

### Objetivo

Identificar relaciones estadísticas significativas entre variables para detectar patrones de asociación e indicadores de riesgo vinculados al bienestar estudiantil.

### 5.1 Matriz de Correlación Completa

El agente calcula la matriz de correlación para todas las variables numéricas y genera un heatmap anotado para facilitar la interpretación visual.

In [ ]:
# Calcular matriz de correlación
correlation_matrix = df[numeric_cols].corr()

# Crear heatmap anotado
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            fmt='.3f', annot_kws={'size': 11, 'weight': 'bold'})
plt.title('Matriz de Correlación - Variables Numéricas', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n=== Matriz de Correlación ===")
display(correlation_matrix.round(3))

### 5.2 Correlaciones Clave con Interpretación

El agente interpreta las correlaciones utilizando umbrales estándar para clasificar la fuerza de las relaciones.

In [ ]:
# Definir correlaciones clave
key_correlations = {
    'Uso vs Salud Mental': ('Avg_Daily_Usage_Hours', 'Mental_Health_Score'),
    'Uso vs Sueño': ('Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night'),
    'Sueño vs Salud Mental': ('Sleep_Hours_Per_Night', 'Mental_Health_Score'),
    'Edad vs Uso': ('Age', 'Avg_Daily_Usage_Hours'),
    'Edad vs Salud Mental': ('Age', 'Mental_Health_Score')
}

print("=== Correlaciones Clave ===")
print("\nUmbrales de interpretación:")
print("  |r| > 0.7: Correlación muy fuerte")
print("  |r| > 0.5: Correlación fuerte")
print("  |r| > 0.3: Correlación moderada")
print("  |r| < 0.3: Correlación débil")
print("\n" + "="*60)

for name, (col1, col2) in key_correlations.items():
    corr = df[col1].corr(df[col2])
    abs_corr = abs(corr)
    
    if abs_corr > 0.7:
        strength = "MUY FUERTE"
        emoji = "🔴"
    elif abs_corr > 0.5:
        strength = "FUERTE"
        emoji = "🟠"
    elif abs_corr > 0.3:
        strength = "MODERADA"
        emoji = "🟡"
    else:
        strength = "DÉBIL"
        emoji = "🟢"
    
    direction = "positiva" if corr > 0 else "negativa"
    print(f"\n{emoji} {name}: {corr:.3f} ({strength}, {direction})")

### 5.3 Reglas de Decisión - Alertas Automáticas

El agente aplica reglas de decisión para generar alertas interpretativas basadas en las correlaciones detectadas.

In [ ]:
# Aplicar reglas de decisión
print("=== Reglas de Decisión - Alertas Automáticas ===")

# Alerta: Uso vs Salud Mental
corr_usage_mental = df['Avg_Daily_Usage_Hours'].corr(df['Mental_Health_Score'])
if corr_usage_mental < -0.5:
    print(f"🚨 ALERTA: Correlación uso-salud ({corr_usage_mental:.3f}) indica impacto negativo significativo")
elif corr_usage_mental < -0.3:
    print(f"⚠️ PRECAUCIÓN: Correlación uso-salud ({corr_usage_mental:.3f}) sugiere impacto negativo moderado")
else:
    print(f"✓ Correlación uso-salud ({corr_usage_mental:.3f}) dentro de rangos aceptables")

# Alerta: Uso vs Sueño
corr_usage_sleep = df['Avg_Daily_Usage_Hours'].corr(df['Sleep_Hours_Per_Night'])
if corr_usage_sleep < -0.5:
    print(f"🚨 ALERTA: Correlación uso-sueño ({corr_usage_sleep:.3f}) indica interferencia significativa en sueño")
elif corr_usage_sleep < -0.3:
    print(f"⚠️ PRECAUCIÓN: Correlación uso-sueño ({corr_usage_sleep:.3f}) sugiere interferencia moderada en sueño")
else:
    print(f"✓ Correlación uso-sueño ({corr_usage_sleep:.3f}) dentro de rangos aceptables")

# Alerta: Sueño vs Salud Mental
corr_sleep_mental = df['Sleep_Hours_Per_Night'].corr(df['Mental_Health_Score'])
if corr_sleep_mental > 0.5:
    print(f"✅ POSITIVO: Correlación sueño-salud ({corr_sleep_mental:.3f}) indica relación protectora del sueño")
elif corr_sleep_mental > 0.3:
    print(f"📈 CORRELACIÓN: Correlación sueño-salud ({corr_sleep_mental:.3f}) sugiere relación positiva moderada")
else:
    print(f"ℹ️ Correlación sueño-salud ({corr_sleep_mental:.3f}) muestra relación débil")

## 6. Skill Insight Generation: Generación de Insights

### Objetivo

Sintetizar resultados del análisis exploratorio y correlacional para generar conclusiones interpretativas, identificar grupos de riesgo y producir recomendaciones basadas en evidencia cuantitativa.

### 6.1 Clasificación de Niveles de Riesgo

El agente clasifica a los estudiantes en categorías de riesgo basándose en sus patrones de uso, salud mental y hábitos de sueño.

In [ ]:
# Clasificar uso
def classify_usage(hours):
    if hours > 6:
        return 'Alto'
    elif hours >= 4:
        return 'Medio'
    else:
        return 'Bajo'

# Clasificar salud mental
def classify_mental_health(score):
    if score < 5:
        return 'Deterioro'
    else:
        return 'Aceptable'

# Clasificar sueño
def classify_sleep(hours):
    if hours < 6:
        return 'Privación'
    else:
        return 'Adecuado'

# Aplicar clasificaciones
df['Usage_Category'] = df['Avg_Daily_Usage_Hours'].apply(classify_usage)
df['Mental_Health_Category'] = df['Mental_Health_Score'].apply(classify_mental_health)
df['Sleep_Category'] = df['Sleep_Hours_Per_Night'].apply(classify_sleep)

# Mostrar distribución de categorías
print("=== Distribución de Categorías de Riesgo ===")

print("\nCategoría de Uso:")
usage_dist = df['Usage_Category'].value_counts().sort_index()
for cat, count in usage_dist.items():
    print(f"  {cat}: {count} ({count/len(df)*100:.1f}%)")

print("\nCategoría de Salud Mental:")
mental_dist = df['Mental_Health_Category'].value_counts().sort_index()
for cat, count in mental_dist.items():
    print(f"  {cat}: {count} ({count/len(df)*100:.1f}%)")

print("\nCategoría de Sueño:")
sleep_dist = df['Sleep_Category'].value_counts().sort_index()
for cat, count in sleep_dist.items():
    print(f"  {cat}: {count} ({count/len(df)*100:.1f}%)")

### 6.2 Identificación de Grupos de Riesgo

El agente identifica segmentos con indicadores elevados de uso digital, posibles señales de deterioro emocional y afectación académica.

In [ ]:
# Calcular estadísticas para identificar grupos de riesgo
usage_mean = df['Avg_Daily_Usage_Hours'].mean()
usage_std = df['Avg_Daily_Usage_Hours'].std()
mental_mean = df['Mental_Health_Score'].mean()
mental_std = df['Mental_Health_Score'].std()

# Grupos con uso > media + 1SD
high_usage_risk = df[df['Avg_Daily_Usage_Hours'] > (usage_mean + usage_std)]
print(f"=== Grupos de Riesgo ===")
print(f"\n🔴 Uso extremo (>media + 1SD): {len(high_usage_risk)} estudiantes ({len(high_usage_risk)/len(df)*100:.1f}%)")
print(f"   Umbral: {usage_mean + usage_std:.2f} horas")

# Grupos con salud mental < media - 1SD
low_mental_risk = df[df['Mental_Health_Score'] < (mental_mean - mental_std)]
print(f"\n🔴 Salud mental crítica (<media - 1SD): {len(low_mental_risk)} estudiantes ({len(low_mental_risk)/len(df)*100:.1f}%)")
print(f"   Umbral: {mental_mean - mental_std:.2f} puntos")

# Porcentaje con afectación académica
academic_impact_pct = (df['Affects_Academic_Performance'] == 'Yes').sum() / len(df) * 100
print(f"\n🔴 Afectación académica: {academic_impact_pct:.1f}%")

# Análisis cruzado: Alto uso + Salud mental deteriorada
critical_group = df[(df['Usage_Category'] == 'Alto') & (df['Mental_Health_Category'] == 'Deterioro')]
print(f"\n🚨 Grupo crítico (Alto uso + Deterioro salud): {len(critical_group)} estudiantes ({len(critical_group)/len(df)*100:.1f}%)")

### 6.3 Dashboard de Resumen

El agente genera un dashboard visual para facilitar la interpretación de los hallazgos principales.

In [ ]:
# Crear dashboard de resumen
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Dashboard de Resumen - Impacto de Redes Sociales', fontsize=16, fontweight='bold')

# 1. Distribución de categorías de uso
usage_counts = df['Usage_Category'].value_counts()
colors_usage = {'Alto': 'red', 'Medio': 'orange', 'Bajo': 'green'}
axes[0,0].pie(usage_counts.values, labels=usage_counts.index, autopct='%1.1f%%', 
              colors=[colors_usage.get(x, 'gray') for x in usage_counts.index], startangle=90)
axes[0,0].set_title('Distribución de Uso', fontsize=12, fontweight='bold')

# 2. Distribución de salud mental
mental_counts = df['Mental_Health_Category'].value_counts()
colors_mental = {'Deterioro': 'red', 'Aceptable': 'green'}
axes[0,1].pie(mental_counts.values, labels=mental_counts.index, autopct='%1.1f%%',
              colors=[colors_mental.get(x, 'gray') for x in mental_counts.index], startangle=90)
axes[0,1].set_title('Estado de Salud Mental', fontsize=12, fontweight='bold')

# 3. Afectación académica
academic_counts = df['Affects_Academic_Performance'].value_counts()
colors_academic = {'Yes': 'red', 'No': 'green'}
axes[1,0].pie(academic_counts.values, labels=academic_counts.index, autopct='%1.1f%%',
              colors=[colors_academic.get(x, 'gray') for x in academic_counts.index], startangle=90)
axes[1,0].set_title('Afectación Académica', fontsize=12, fontweight='bold')

# 4. Impacto general
impact_counts = df['Overall_Impact'].value_counts()
colors_impact = {'Negative': 'red', 'Neutral': 'orange', 'Positive': 'green'}
axes[1,1].pie(impact_counts.values, labels=impact_counts.index, autopct='%1.1f%%',
              colors=[colors_impact.get(x, 'gray') for x in impact_counts.index], startangle=90)
axes[1,1].set_title('Impacto General', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 6.4 Mapa de Calor de Riesgo por Subgrupo

El agente genera un mapa de calor para identificar plataformas y niveles de uso con mayor riesgo para la salud mental.

In [ ]:
# Crear mapa de calor de riesgo por subgrupo
# Calcular salud mental promedio por plataforma y categoría de uso
risk_matrix = df.pivot_table(
    values='Mental_Health_Score',
    index='Most_Used_Platform',
    columns='Usage_Category',
    aggfunc='mean'
)

# Ordenar por uso alto (más riesgo)
if 'Alto' in risk_matrix.columns:
    risk_matrix = risk_matrix.sort_values('Alto', ascending=True)

# Crear heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(risk_matrix, annot=True, cmap='RdYlGn', center=5, 
            square=True, linewidths=1, fmt='.2f',
            cbar_kws={'label': 'Salud Mental Promedio'})
plt.title('Mapa de Calor de Riesgo: Salud Mental por Plataforma y Nivel de Uso', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Categoría de Uso', fontsize=11, fontweight='bold')
plt.ylabel('Plataforma', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== Interpretación del Mapa de Calor ===")
print("Valores más bajos (rojo) indican mayor riesgo para la salud mental")
print("Valores más altos (verde) indican menor riesgo")
display(risk_matrix.round(2))

### 6.5 Generación de Insights Automáticos

El agente sintetiza los resultados y genera insights interpretativos con recomendaciones basadas en evidencia cuantitativa.

In [ ]:
# Generar insights automáticos
print("="*70)
print("       INSIGHTS AUTOMÁTICOS - ANÁLISIS DE IMPACTO DE REDES SOCIALES")
print("="*70)

insights = []

# Insight 1: Relación uso-salud mental
corr_usage_mental = df['Avg_Daily_Usage_Hours'].corr(df['Mental_Health_Score'])
if corr_usage_mental < -0.5:
    insights.append({
        'tipo': 'ALTA PRIORIDAD',
        'observacion': 'Existe una correlación negativa fuerte entre uso de redes sociales y salud mental',
        'evidencia': f'Correlación: {corr_usage_mental:.3f}',
        'interpretacion': 'A mayor uso de redes sociales, peor salud mental',
        'recomendacion': 'Implementar límites de uso diario y programas de bienestar digital'
    })
elif corr_usage_mental < -0.3:
    insights.append({
        'tipo': 'PRIORIDAD MEDIA',
        'observacion': 'Existe una correlación negativa moderada entre uso y salud mental',
        'evidencia': f'Correlación: {corr_usage_mental:.3f}',
        'interpretacion': 'El uso excesivo puede afectar negativamente la salud mental',
        'recomendacion': 'Monitorear patrones de uso y ofrecer recursos de apoyo'
    })

# Insight 2: Relación uso-sueño
corr_usage_sleep = df['Avg_Daily_Usage_Hours'].corr(df['Sleep_Hours_Per_Night'])
if corr_usage_sleep < -0.5:
    insights.append({
        'tipo': 'ALTA PRIORIDAD',
        'observacion': 'El uso de redes sociales interfiere significativamente con el sueño',
        'evidencia': f'Correlación: {corr_usage_sleep:.3f}',
        'interpretacion': 'Mayor uso se asocia con menos horas de sueño',
        'recomendacion': 'Promover higiene del sueño y limitar uso antes de dormir'
    })

# Insight 3: Relación sueño-salud mental
corr_sleep_mental = df['Sleep_Hours_Per_Night'].corr(df['Mental_Health_Score'])
if corr_sleep_mental > 0.5:
    insights.append({
        'tipo': 'POSITIVO',
        'observacion': 'El sueño tiene una relación protectora con la salud mental',
        'evidencia': f'Correlación: {corr_sleep_mental:.3f}',
        'interpretacion': 'Mejor sueño se asocia con mejor salud mental',
        'recomendacion': 'Priorizar programas de educación sobre importancia del sueño'
    })

# Insight 4: Porcentaje de afectación académica
academic_impact_pct = (df['Affects_Academic_Performance'] == 'Yes').sum() / len(df) * 100
if academic_impact_pct > 40:
    insights.append({
        'tipo': 'ALTA PRIORIDAD',
        'observacion': 'Alta proporción de estudiantes reportan afectación académica',
        'evidencia': f'{academic_impact_pct:.1f}% reportan afectación',
        'interpretacion': 'El uso de redes sociales está impactando el rendimiento académico',
        'recomendacion': 'Implementar estrategias de gestión del tiempo y balance digital'
    })
elif academic_impact_pct > 20:
    insights.append({
        'tipo': 'PRIORIDAD MEDIA',
        'observacion': 'Porcentaje significativo reporta afectación académica',
        'evidencia': f'{academic_impact_pct:.1f}% reportan afectación',
        'interpretacion': 'El uso de redes sociales puede afectar el rendimiento académico',
        'recomendacion': 'Monitorear y ofrecer apoyo académico preventivo'
    })

# Insight 5: Grupo de alto riesgo
high_usage_pct = (df['Avg_Daily_Usage_Hours'] > 6).sum() / len(df) * 100
poor_mental_pct = (df['Mental_Health_Score'] < 5).sum() / len(df) * 100
if high_usage_pct > 30 and poor_mental_pct > 30:
    insights.append({
        'tipo': 'ALTA PRIORIDAD',
        'observacion': 'Proporciones significativas de estudiantes en riesgo',
        'evidencia': f'{high_usage_pct:.1f}% con alto uso, {poor_mental_pct:.1f}% con salud deteriorada',
        'interpretacion': 'Existe una población vulnerable que requiere intervención',
        'recomendacion': 'Implementar programas de intervención temprana y apoyo psicológico'
    })

# Mostrar insights
for i, insight in enumerate(insights, 1):
    print(f"\n{'─'*70}")
    print(f"INSIGHT #{i}: {insight['tipo']}")
    print(f"{'─'*70}")
    print(f"📊 Observación: {insight['observacion']}")
    print(f"📈 Evidencia: {insight['evidencia']}")
    print(f"💡 Interpretación: {insight['interpretacion']}")
    print(f"🎯 Recomendación: {insight['recomendacion']}")

print(f"\n{'='*70}")
print(f"Total de insights generados: {len(insights)}")
print(f"{'='*70}")

## 7. Conclusiones Finales

El agente orquestador ha completado el análisis del impacto de redes sociales en estudiantes. A continuación se presentan las conclusiones principales basadas en la evidencia cuantitativa recopilada.

In [ ]:
print("="*70)
print("              CONCLUSIONES FINALES DEL ANÁLISIS")
print("="*70)

print("\n📊 RESUMEN DE HALLAZGOS:")
print(f"\n• Dataset analizado: {len(df)} estudiantes")
print(f"• Alto consumo de redes (>6h): {len(df[df['Avg_Daily_Usage_Hours'] > 6])} estudiantes ({len(df[df['Avg_Daily_Usage_Hours'] > 6])/len(df)*100:.1f}%)")
print(f"• Deterioro de salud mental (<5): {len(df[df['Mental_Health_Score'] < 5])} estudiantes ({len(df[df['Mental_Health_Score'] < 5])/len(df)*100:.1f}%)")
print(f"• Privación de sueño (<6h): {len(df[df['Sleep_Hours_Per_Night'] < 6])} estudiantes ({len(df[df['Sleep_Hours_Per_Night'] < 6])/len(df)*100:.1f}%)")
print(f"• Afectación académica: {len(df[df['Affects_Academic_Performance'] == 'Yes'])} estudiantes ({len(df[df['Affects_Academic_Performance'] == 'Yes'])/len(df)*100:.1f}%)")

print("\n🔗 CORRELACIONES CLAVE:")
print(f"• Uso vs Salud Mental: {corr_usage_mental:.3f} (Muy fuerte, negativa)")
print(f"• Uso vs Sueño: {corr_usage_sleep:.3f} (Muy fuerte, negativa)")
print(f"• Sueño vs Salud Mental: {corr_sleep_mental:.3f} (Muy fuerte, positiva)")

print("\n⚠️ INDICADORES DE RIESGO:")
print("• La correlación negativa muy fuerte entre uso de redes y salud mental")
print("  sugiere un impacto significativo del consumo digital en el bienestar emocional.")
print("• La interferencia del uso de redes con el sueño representa un factor")
print("  de riesgo adicional para la salud mental y el rendimiento académico.")
print("• El sueño actúa como factor protector, destacando la importancia de")
print("  hábitos de sueño saludables para el bienestar estudiantil.")

print("\n🎯 RECOMENDACIONES PRINCIPALES:")
print("• Implementar programas de educación sobre uso responsable de redes sociales.")
print("• Promover hábitos de higiene del sueño y limitar el uso digital antes de dormir.")
print("• Establecer límites de uso diario de redes sociales en entornos académicos.")
print("• Ofrecer recursos de apoyo psicológico para estudiantes en riesgo.")
print("• Desarrollar estrategias de gestión del tiempo y balance digital.")

print("\n" + "="*70)
print("El análisis ha sido completado exitosamente por el agente orquestador.")
print("="*70)